<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/pytorch-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="text-align: right; flex: 1;">
        <a href="https://pytorch.org/" target="_blank" rel="noopener noreferrer">
            <img src="../../assets/images/pytorch/logo/pytorch-logo-dark.svg" 
                 alt="PyTorch Logo"
                 style="max-height: 48px; width: auto; background-color: #ffffff; border-radius: 8px;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Word Embedding using Word2Vec [Skip-Grams method]](#toc2_)    
  - [Hyperparameters](#toc2_1_)    
  - [Load and preprocess the dataset](#toc2_2_)    
  - [Create Dataset and DataLoader](#toc2_3_)    
  - [Define and Initialize the Word2Vec model](#toc2_4_)    
  - [Training Loop](#toc2_5_)    
  - [Save embeddings](#toc2_6_)    
  - [Visualize word embeddings using t-SNE](#toc2_7_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary

from datasets import load_dataset

In [ ]:
# disable automatic figure display (plt.show() required)  
# this ensures consistency with .py scripts and gives full control over when plots appear
plt.ioff()

In [2]:
# set a seed for deterministic results
seed = 42
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# check if cuda is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# log
device

In [ ]:
# update paths as needed based on your project structure
DATASET_DIR = Path("../../datasets/")
CHECKPOINT_DIR = Path("../../assets/checkpoints/embeddings")

# <a id='toc2_'></a>[Word Embedding using Word2Vec [Skip-Grams method]](#toc0_)

🌟 **Example**:

- **Sentence**: The quick brown fox jumps over the lazy dog.
  - **WINDOW_SIZE: 2**
    <table style="margin:0 auto; text-align:center;">
      <tr >
        <th colspan="2">Left Contexts</th>
        <th rowspan="2">Center Word</th>
        <th colspan="2">Right Contexts</th>
      </tr>
      <tr>
        <th>Left 2</th>
        <th>Left 1</th>
        <th>Right 1</th>
        <th>Right 2</th>
      </tr>
      <tr>
        <td></td>
        <td></td>
        <td>The</td>
        <td>quick</td>
        <td>brown</td>
      </tr>
      <tr>
        <td></td>
        <td>The</td>
        <td>quick</td>
        <td>brown</td>
        <td>fox</td>
      </tr>
      <tr>
        <td>The</td>
        <td>quick</td>
        <td>brown</td>
        <td>fox</td>
        <td>jumps</td>
      </tr>
      <tr>
        <td>quick</td>
        <td>brown</td>
        <td>fox</td>
        <td>jumps</td>
        <td>over</td>
      </tr>
      <tr>
        <td>brown</td>
        <td>fox</td>
        <td>jumps</td>
        <td>over</td>
        <td>the</td>
      </tr>
      <tr>
        <td>fox</td>
        <td>jumps</td>
        <td>over</td>
        <td>the</td>
        <td>lazy</td>
      </tr>
      <tr>
        <td>jumps</td>
        <td>over</td>
        <td>the</td>
        <td>lazy</td>
        <td>dog</td>
      </tr>
      <tr>
        <td>over</td>
        <td>the</td>
        <td>lazy</td>
        <td>dog</td>
        <td></td>
      </tr>
      <tr>
        <td>the</td>
        <td>lazy</td>
        <td>dog</td>
        <td></td>
        <td></td>
      </tr>
    </table>
  - **NEGATIVE_SAMPLES: 5**
    - If we are predicting "brown" and the context words are ["quick", "The", "fox", "jumps"].
    - we will also randomly select 5 words from the rest of the vocabulary (e.g., "lazy", "dog", etc.) and treat them as negative samples.


## <a id='toc2_1_'></a>[Hyperparameters](#toc0_)


In [5]:
EMBEDDING_DIM = 100
WINDOW_SIZE = 2
BATCH_SIZE = 2048
EPOCHS = 2
LEARNING_RATE = 0.01
NEGATIVE_SAMPLES = 5

## <a id='toc2_2_'></a>[Load and preprocess the dataset](#toc0_)


In [ ]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", cache_dir=DATASET_DIR)
train_text = dataset["train"]["text"]

# log
print(f"len(train_text) : {len(train_text)}")
print(f"train_text[0]   : {train_text[0]}")
print(f"train_text[4]   : {train_text[4]}")

In [ ]:
# tokenize and build vocabulary
tokenized_sentences = [sentence.split() for sentence in train_text if sentence.strip()]
words = [word for sentence in tokenized_sentences for word in sentence]
vocab = Counter(words)

# log
print(f"vocabulary size             : {len(vocab)}")
print(f"top 5 most  frequent tokens : {vocab.most_common(5)}")
print(f"top 5 least frequent tokens : {sorted(vocab.items(), key=lambda x: x[1])[:5]}")

In [ ]:
# filter out rare words
min_freq = 5
vocab = {word: freq for word, freq in vocab.items() if freq >= min_freq}
word2idx = {word: idx for idx, (word, _) in enumerate(vocab.items())}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)

# log
print(f"vocabulary size             : {vocab_size}")
print(f"top 5 most  frequent tokens : {sorted(vocab.items(), key=lambda x: x[1], reverse=True)[:5]}")
print(f"top 5 least frequent tokens : {sorted(vocab.items(), key=lambda x: x[1])[:5]}")

In [ ]:
# generate skip-grams
def generate_skip_grams(
    words: list[str],
    word2idx: dict[str, int],
    window_size: int,
) -> list[tuple[int, int]]:
    pairs = []
    for idx, word in enumerate(words):
        if word not in word2idx:
            continue
        center_idx = word2idx[word]
        context_range = range(max(0, idx - window_size), min(len(words), idx + window_size + 1))
        for context_idx in context_range:
            if context_idx == idx:
                continue
            context_word = words[context_idx]
            if context_word in word2idx:
                pairs.append((center_idx, word2idx[context_word]))
    return pairs


# generate skip-grams
skip_grams = generate_skip_grams(words, word2idx, WINDOW_SIZE)

# log
print(f"total number of skip-grams         : {len(skip_grams)}")
print(f"sequence of words [1000 to 1006]   : {words[1000:1007]}")
print("{word:idx} for the above sequence] :", {w: word2idx[w] for w in words[1000:1007]})
print(
    f"skip-gram pairs for above sequence : {[skip_grams[skip_grams.index((401, 200)) + i] for i in range(WINDOW_SIZE * 2)]}"
)

## <a id='toc2_3_'></a>[Create Dataset and DataLoader](#toc0_)


In [10]:
class SkipGramDataset(Dataset):
    def __init__(self, skip_grams: list[tuple[int, int]], vocab_size: int, neg_samples: int):
        self.skip_grams = skip_grams
        self.vocab_size = vocab_size
        self.neg_samples = neg_samples

    def __len__(self) -> int:
        return len(self.skip_grams)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        center, context = self.skip_grams[idx]
        negatives = torch.multinomial(torch.tensor([1.0] * self.vocab_size), self.neg_samples, replacement=True)
        return torch.tensor(center, dtype=torch.long), torch.tensor(context, dtype=torch.long), negatives

In [ ]:
dataset = SkipGramDataset(skip_grams, vocab_size, NEGATIVE_SAMPLES)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
num_batches = len(data_loader)

# log
for i in range(WINDOW_SIZE * 2):
    temp_sample = dataset[skip_grams.index((401, 200)) + i]
    print(temp_sample)
    print(f"  negative samples:")
    for j in range(NEGATIVE_SAMPLES):
        print("    ", {temp_sample[2][j].item(): idx2word[temp_sample[2][j].item()]})
    print()

## <a id='toc2_4_'></a>[Define and Initialize the Word2Vec model](#toc0_)


In [12]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, center: torch.Tensor, context: torch.Tensor, negatives: torch.Tensor) -> torch.Tensor:
        center_embed = self.embeddings(center)
        context_embed = self.context_embeddings(context)
        neg_embed = self.context_embeddings(negatives)

        # positive scores
        positive_score = torch.sum(center_embed * context_embed, dim=1)
        positive_loss = -F.logsigmoid(positive_score)

        # negative scores
        negative_score = torch.bmm(neg_embed, center_embed.unsqueeze(2)).squeeze(2)
        negative_loss = -torch.sum(F.logsigmoid(-negative_score), dim=1)

        return (positive_loss + negative_loss).mean()

In [13]:
model = Word2Vec(vocab_size, EMBEDDING_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
model

In [ ]:
summary(
    model,
    input_data=(
        (
            torch.randint(0, vocab_size, (BATCH_SIZE,)).to(device),
            torch.randint(0, vocab_size, (BATCH_SIZE,)).to(device),
            torch.randint(0, vocab_size, (BATCH_SIZE, NEGATIVE_SAMPLES)).to(device),
        )
    ),
)

## <a id='toc2_5_'></a>[Training Loop](#toc0_)


In [ ]:
for epoch in range(EPOCHS):
    total_loss = 0
    model.train()

    # log
    print(f"epoch {epoch + 1:>0{len(str(EPOCHS))}}/{EPOCHS}")

    for i, (center, context, negatives) in enumerate(data_loader):
        # move data to the correct device
        center, context, negatives = center.to(device), context.to(device), negatives.to(device)

        optimizer.zero_grad()
        loss = model(center, context, negatives)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # log
        if i % 100 == 0:
            print(
                f"  iteration {i + 1:>0{len(str(num_batches))}}/{num_batches}  |  loss: {loss.item():.5f}  |  total loss: {total_loss:.5f}"
            )

    # log
    print(f"epoch {epoch+1:>0{len(str(EPOCHS))}}/{EPOCHS}  |  total loss: {total_loss:.5f}")
    print("-" * 50)

## <a id='toc2_6_'></a>[Save embeddings](#toc0_)


In [ ]:
embeddings = model.embeddings.weight.data
torch.save(embeddings, CHECKPOINT_DIR / "word2vec_embeddings.pt")

# log
print()

## <a id='toc2_7_'></a>[Visualize word embeddings using t-SNE](#toc0_)


In [ ]:
def visualize_embeddings(embeddings: torch.Tensor, idx2word: dict[int, str], num_points: int = 100) -> None:
    tsne = TSNE(n_components=2, random_state=seed)
    reduced_embeddings = tsne.fit_transform(embeddings[:num_points].cpu().numpy())

    plt.figure(figsize=(10, 10))
    for idx, (x, y) in enumerate(reduced_embeddings):
        plt.scatter(x, y)
        plt.text(x, y, idx2word[idx], fontsize=9)
    plt.show()

In [ ]:
visualize_embeddings(embeddings, idx2word)